In [11]:
import pandas as pd
import numpy as np

In [12]:
x_train = pd.read_csv('trainNaive.csv')
y_train = pd.read_csv('trainNaiveLabels.csv', header=None, skiprows=1)
y_train = y_train.iloc[:, 0].values

In [25]:
class NaiveBayesClassifier:
    def __init__(self):
        self.classProb = {}
        self.featProb = {}
        self.classes = []

    def fit(self, x_train, y_train):
        n_samples, n_features = x_train.shape
        self.classes = np.unique(y_train)
        n_classes = len(self.classes)

        for cla in self.classes:
            self.classProb[cla] = np.sum(y_train == cla) / n_samples
            self.featProb[cla] = []

            for featIndex in range(n_features):
                unique_feature_values = np.unique(x_train.iloc[:, featIndex])
                featProbability = {}

                for featValue in unique_feature_values:
                    count_with_value = np.sum((x_train.iloc[:, featIndex] == featValue) & (y_train == cla))
                    count_total = np.sum(y_train == cla)
                    smoothed_prob = (count_with_value + 1) / (count_total + len(unique_feature_values))
                    featProbability[featValue] = smoothed_prob
                self.featProb[cla].append(featProbability)

    def predict(self, X_test):
        predictions = []

        for x in X_test.values:
            probs = []

            for c in self.classes:
                class_prob = self.classProb[c]
                feature_prob = 1

                for featIndex, featValue in enumerate(x):
                    if featValue in self.featProb[c][featIndex]:
                        feature_prob *= self.featProb[c][featIndex][featValue]
                    else:
                        feature_prob *= 1 / (np.sum(y_train == c) + len(np.unique(x_train.iloc[:, featIndex])))

                probs.append(class_prob * feature_prob)
            predictions.append(self.classes[np.argmax(probs)])

        return predictions


In [26]:
classes = np.unique(y_train)

classifier = NaiveBayesClassifier()
classifier.fit(x_train, y_train)

x_test = pd.read_csv('testNaive.csv')

predictions = classifier.predict(x_test)
print(predictions)

['yes', 'yes', 'yes', 'yes', 'no']
